# 

The Maxwell equations in vaccum are:
\begin{eqnarray}
  \partial_t {\bf D} &=& {\bf \nabla}\times{\bf B},\\
  \partial_t {\bf B} &=& -{\bf \nabla}\times{\bf D},\\
  {\bf \nabla}\cdot{\bf B} &=& 0,\\
  {\bf \nabla}\cdot{\bf D} &=& 0,\\
\end{eqnarray}
where ${\bf D} = {\bf E}$, $\mu = 1$, $\epsilon=1$, and $\sigma = 0$.

If a vector field ${\bf A}$ satisfies
$\partial^2_t A  - \nabla^2 {\bf A} = 0$ and ${\bf \nabla}\cdot{\bf A} = 0$,
then
${\bf E} = \partial_t {\bf A}$ and ${\bf B} = {\bf \nabla}\times {\bf A}$ satisfy the Maxwell Equations

In [1]:
import sys
from sympy import var, Function, Derivative, Wild, Symbol, pi, symbols, Subs
from sympy import Eq, zeros, Matrix, solve, IndexedBase, Idx, ccode,exp, cos, sin, sqrt, diff
from sympy.vector import CoordSys3D, Del, curl, divergence
import re
Cart = CoordSys3D('Cart')
delop = Del()
pi = var('pi')
t = var('t')
xx = Cart.x
yy = Cart.y
zz = Cart.z
nx = Cart.i
ny = Cart.j
nz = Cart.k


In [2]:
ax, ay, k = var(('ax', 'ay', 'k'))

f, df, ddf = Function('f'), Function('df'), Function('ddf')
Ax = ax * sin(k*(zz-t)) * f(t-zz)
Ay = ay * cos(k*(zz-t)) * f(t-zz)
Az = 0

Avec= Ax * nx + Ay * ny + Az * nz

In [3]:
Evec = - diff(Avec, t)
Bvec = curl(Avec, t)
print("Checking Maxwell's equations")
print(" div E =", divergence(Evec).simplify())
print(" div B =", divergence(Bvec).simplify())
print(" dt_E - curl B =", (diff(Evec, t) - curl(Bvec)).simplify())
print(" dt_B + curl E =", (diff(Bvec, t) + curl(Evec)).simplify())

Checking Maxwell's equations
 div E = 0
 div B = 0
 dt_E - curl B = 0
 dt_B + curl E = 0


In [4]:
free1=diff(Evec,t).dot(nx) + diff(Bvec, t).dot(ny)
free2=diff(Evec,t).dot(ny) - diff(Bvec, t).dot(nx)

In [5]:
x, y, z = var(('x', 'y', 'z'))

a, b = Wild('a'), Wild('b')
Evec = Evec.replace(Derivative(b, a), df(a)).simplify()
Bvec = Bvec.replace(Derivative(b, a), df(a)).simplify()

E_coor = Evec.subs({xx : x, yy : y, zz : z})
B_coor = Bvec.subs({xx : x, yy : y, zz : z})

free1 = free1.replace(Derivative(b, a), df(a)).simplify()
free1 = free1.replace(Derivative(b, a,2), ddf(a)).simplify()
free2 = free2.replace(Derivative(b, a), df(a)).simplify()
free2 = free2.replace(Derivative(b, a, 2), ddf(a)).simplify()
free1 = free1.subs({xx : x, yy : y, zz : z})
free2 = free2.subs({xx : x, yy : y, zz : z})

Ex = E_coor.dot(nx)
Ey = E_coor.dot(ny)
Ez = E_coor.dot(nz)

Bx = B_coor.dot(nx).simplify()
By = B_coor.dot(ny).simplify()
Bz = B_coor.dot(nz).simplify()

custom_functions={"f" : "f", "df" : "df", "ddf" : "ddf"}
print('void standing_wave(const int l, const int m, const int n, const double ax,\n'+
      '                   const double ay, const double x, const double y,\n'+
      '                   const double z, const double t, struct eb_st *A)')
print('{')

print('  A->Dx = ', ccode(Ex, user_functions=custom_functions), ';', sep='')
print('  A->Dy = ', ccode(Ey, user_functions=custom_functions),';', sep='')
print('  A->Dz = ', ccode(Ez, user_functions=custom_functions), ';', sep='')

print('  A->Bx = ', ccode(Bx, user_functions=custom_functions), ';', sep='')
print('  A->By = ', ccode(By, user_functions=custom_functions),';', sep='')
print('  A->Bz = ', ccode(Bz, user_functions=custom_functions), ';', sep='')
print('  A->PsiD = 0;')
print('  A->PsiB = 0;')
print('  A->rho = 0;')
print('}')



void standing_wave(const int l, const int m, const int n, const double ax,
                   const double ay, const double x, const double y,
                   const double z, const double t, struct eb_st *A)
{
  A->Dx = ax*(k*f(t - z)*cos(k*(-t + z)) - df(t - z)*sin(k*(-t + z)));
  A->Dy = ay*(-k*f(t - z)*sin(k*(-t + z)) - df(t - z)*cos(k*(-t + z)));
  A->Dz = 0;
  A->Bx = ay*(-k*f(t - z)*sin(k*(t - z)) + df(t - z)*cos(k*(t - z)));
  A->By = ax*(k*f(t - z)*cos(k*(t - z)) + df(t - z)*sin(k*(t - z)));
  A->Bz = 0;
  A->PsiD = 0;
  A->PsiB = 0;
  A->rho = 0;
}


In [6]:
print(ccode(free1, user_functions=custom_functions))

2*ax*(pow(k, 2)*f(t - z)*sin(k*(-t + z)) + 2*k*df(t - z)*cos(k*(-t + z)) - ddf(t - z)*sin(k*(-t + z)))


In [7]:
print(ccode(free2, user_functions=custom_functions))

2*ay*(pow(k, 2)*f(t - z)*cos(k*(-t + z)) - 2*k*df(t - z)*sin(k*(-t + z)) - ddf(t - z)*cos(k*(-t + z)))


In [8]:
a, b, c, xa, xb = var('a, b, c, xa, xb')

In [9]:
f=(x-a)**4/(b-a)**4 * (1 + -4 / (b-a)* (x - b) + 10/(b-a)**2 * (x-b)**2  - 20/(b-a)**3 * (x-b)**3)

In [16]:
print(f.subs({x :a}))
print(diff(f,x,1).subs({x :a}))
print(diff(f,x,2).subs({x :a}))
print(diff(f,x,3).subs({x :a}))
print(f.subs({x :b}))
print(diff(f,x,1).subs({x :b}))
print(diff(f,x,2).subs({x :b}))
print(diff(f,x,3).subs({x :b}))


0
0
0
0
1
0
0
0


In [17]:
xa2, xa3, xa4 = var('xa2 xa3 xa4')
xb2, xb3, xb4 = var('xb2 xb3 xb4')

In [18]:
print(ccode(f.subs(
    {b-a: c, x-a: c * xa, x-b : c * xb}
    ).subs({2*x-2*b: 2*c*xb}).simplify().subs(
    {
        xa**4:xa4, xb**3: xb3, xb**2: xb2
    })))


xa4*(-4*xb + 10*xb2 - 20*xb3 + 1)


In [19]:
print(ccode((c*diff(f,x, 1)).subs({b-a: c, x-a: c * xa, x-b : c * xb}).subs(
    {2*x-2*b: 2*c*xb}).simplify().subs(
    {
       xa**2 : xa2, xa**3: xa3,  xa**4:xa4, xb**3: xb3, xb**2: xb2
        
    })))


4*xa3*(xa*(5*xb - 15*xb2 - 1) - 4*xb + 10*xb2 - 20*xb3 + 1)


In [20]:
print(ccode((c**2*diff(f,x, 2)).subs({b-a: c, x-a: c * xa, x-b : c * xb}).subs(
    {2*x-2*b: 2*c*xb}).simplify().subs(
    {
       xa**2 : xa2, xa**3: xa3,  xa**4:xa4, xb**3: xb3, xb**2: xb2
        
    })))


xa2*(-32*xa*(-5*xb + 15*xb2 + 1) + 4*xa2*(5 - 30*xb) - 48*xb + 120*xb2 - 240*xb3 + 12)


In [32]:
print(ccode((c**2*diff(f,x, 2)).subs({b-a: c, x-a: c * xa, x-b : c * xb}).subs(
    {2*x-2*b: 2*c*xb}).simplify().expand().subs(
    {
       xa**3: xa3,  xa**4:xa4, xb**3: xb3, xb**2: xb2
        
    }).subs({xa**2 : xa2}).simplify()))

-48*xa2*xb + 120*xa2*xb2 - 240*xa2*xb3 + 12*xa2 + 160*xa3*xb - 480*xa3*xb2 - 32*xa3 - 120*xa4*xb + 20*xa4
